<a href="https://colab.research.google.com/github/Safwanach2002/ML/blob/main/PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# PRINCIPAL COMPONENT ANALYSIS (PCA)

# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

# 1. Load the Dataset
uploaded = files.upload()
df = pd.read_excel(list(uploaded.keys())[0])

features = [f"Feature_{i}" for i in range(1, 7)]
X = df[features]
y = df["Category"]

print(df.head(10))
print("Dataset Shape:", df.shape)
print("Features:", features)
print("Target: Category")
print(y.value_counts())

# Plot Category Distribution
sns.countplot(x=y)
plt.title("Category Distribution")
plt.show()

# 2. Standardize the Features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_df = pd.DataFrame(X_scaled, columns=features)
print("Standardized Data:")
print(scaled_df.head())

# Plot Before and After Scaling
plt.figure(figsize=(10, 4))
sns.boxplot(data=X)
plt.title("Before Standardization")
plt.show()

plt.figure(figsize=(10, 4))
sns.boxplot(data=scaled_df)
plt.title("After Standardization")
plt.show()

# 3. Calculate and Visualize Covariance Matrix
cov_matrix = np.cov(X_scaled, rowvar=False)
cov_df = pd.DataFrame(cov_matrix, index=features, columns=features)

print("Covariance Matrix:")
print(cov_df.round(2))

sns.heatmap(cov_df, annot=True, cmap="coolwarm")
plt.title("Covariance Matrix")
plt.show()

# 4. Calculate Eigenvalues and Eigenvectors
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)

# Sort in descending order
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

# Explained Variance
variance = eigenvalues / sum(eigenvalues)
cumulative = np.cumsum(variance)

print("Eigenvalues:", eigenvalues.round(3))
print("Eigenvectors:\n", eigenvectors.round(3))
print("Explained Variance:", variance.round(3))
print("Cumulative Variance:", cumulative.round(3))

# Components required for 95% variance
print("Components for 95% variance:",
      np.argmax(cumulative >= 0.95) + 1)

# 5. Apply PCA Using Scikit-learn
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
print("PCA Data:")
print(pca_df.head(10))

print("Variance Ratio:", pca.explained_variance_ratio_)
print("Total Variance:",
      pca.explained_variance_ratio_.sum() * 100, "%")

# 6. Visualize PCA Data
pca_df["Category"] = y.values

sns.scatterplot(data=pca_df, x="PC1", y="PC2",
                hue="Category")
plt.title("PCA Transformed Data")
plt.show()

# 7. Analyze PCA Loadings
loadings = pd.DataFrame(
    pca.components_.T,
    index=features,
    columns=["PC1", "PC2"]
)

print("PCA Loadings:")
print(loadings.round(3))

sns.heatmap(loadings, annot=True, cmap="coolwarm")
plt.title("PCA Loadings")
plt.show()

# 8. Reconstruct Original Data
X_reconstructed = scaler.inverse_transform(
    pca.inverse_transform(X_pca)
)

print("Original Data:", X.iloc[0].values)
print("Reconstructed Data:", X_reconstructed[0].round(2))

mse = mean_squared_error(X, X_reconstructed)
print("Reconstruction MSE:", mse)

# 9. Reconstruction Error for Different Components
errors = []

for n in range(1, 7):
    pca_temp = PCA(n_components=n)
    transformed = pca_temp.fit_transform(X_scaled)
    reconstructed = pca_temp.inverse_transform(transformed)
    errors.append(mean_squared_error(X_scaled, reconstructed))

print("Reconstruction Errors:", errors)

plt.plot(range(1, 7), errors, marker='o')
plt.xlabel("Number of Components")
plt.ylabel("Reconstruction MSE")
plt.title("Components vs Reconstruction Error")
plt.show()

# 10. Effect of PCA on KNN Classification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# KNN with Original Features
knn1 = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

knn1.fit(X_train, y_train)
pred1 = knn1.predict(X_test)
acc1 = accuracy_score(y_test, pred1)

# KNN with PCA Features
knn2 = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

knn2.fit(X_train, y_train)
pred2 = knn2.predict(X_test)
acc2 = accuracy_score(y_test, pred2)

print("KNN Original Accuracy:", acc1 * 100, "%")
print("KNN PCA Accuracy:", acc2 * 100, "%")

# Plot Accuracy Comparison
plt.bar(["Original", "PCA"], [acc1 * 100, acc2 * 100])
plt.ylabel("Accuracy (%)")
plt.title("KNN Accuracy Comparison")
plt.show()

# 11. Summary
print("PCA SUMMARY")
print("Original Features:", len(features))
print("Reduced Features: 2")
print("Variance Retained:",
      pca.explained_variance_ratio_.sum() * 100, "%")
print("Reconstruction MSE:", mse)
print("Original KNN Accuracy:", acc1 * 100, "%")
print("PCA KNN Accuracy:", acc2 * 100, "%")

print("PCA reduces features while retaining important information.")
